In [2]:
import pandas as pd
import numpy as np

---

## Level 1 — Library book catalog

A city library exported their catalog before a system migration. The data has four problems:

1. Two books appear twice — exact duplicate rows.
2. `genre` is inconsistently cased (`"Sci-Fi"`, `"SCI-FI"`, `"sci-fi"`). Standardize to lowercase.
3. `status` has five spellings of two values. Normalize: `"available"`, `"Available"`, `"AVAILABLE"` → `"available"`; `"on-loan"`, `"on loan"` → `"on_loan"`.
4. `pages` uses `0` to mean "unknown". Replace with `NaN`.

**New this week — `df.replace()`**

Replaces exact values in a Series or DataFrame. Unlike `.str.replace()` (which runs regex on string content), `.replace()` matches the whole value:

```python
# Map several aliases to one standard label
s.replace({'on-loan': 'on_loan', 'on loan': 'on_loan'})

# Replace a sentinel number with NaN
s.replace(0, np.nan)
```

After cleaning, answer:
- How many books remain?
- What fraction are currently on loan? Use `np.mean` on a boolean comparison.
- What is the median page count, excluding unknowns? Use `np.nanmedian`.
- How many books are in each genre?

In [22]:
catalog = pd.DataFrame({
    'book_id': ['B001','B002','B003','B004','B005','B006','B007','B008','B003','B006'],
    'genre':   ['Sci-Fi','mystery','THRILLER','Romance','SCI-FI','Mystery','thriller','romance','THRILLER','Mystery'],
    'pages':   [342, 287, 410, 198, 523, 0, 365, 0, 410, 0],
    'status':  ['available','on-loan','Available','on loan','AVAILABLE','on_loan','on-loan','available','Available','on_loan'],
    'checked_out': ['n/a','2026-05-01','n/a','2026-04-28','n/a','2026-05-10','2026-05-03','n/a','n/a','2026-05-10'],
})

# Your code here
catalog['status'] = catalog['status'].replace({'on-loan':'on_loan','on loan':'on_loan'})
catalog['status'] = catalog['status'].str.lower()
catalog = catalog.drop_duplicates()
catalog['genre'] = catalog['genre'].str.lower()
catalog['pages'] = catalog['pages'].replace(0, np.nan)

print(catalog.shape[0],'books remain')
print(np.mean(catalog['status']=='on_loan'),'are currently on loan')
print('median page count is: ', np.nanmedian(catalog['pages']))
print(catalog['genre'].value_counts())


8 books remain
0.5 are currently on loan
median page count is:  353.5
genre
sci-fi      2
mystery     2
thriller    2
romance     2
Name: count, dtype: int64


---

## Level 2 — Air quality sensor network

Four sensors recorded temperature and AQI (Air Quality Index) across three months. Known data quirks:
- `temp_c`: `-999` indicates a sensor failure — no reading was recorded
- `aqi`: `'ERR'` appears when the air quality meter malfunctioned
- `status_code`: 1 = normal, 2 = maintenance, 3 = offline — replace codes with string labels

Clean the data — no steps provided — then answer:

- Which sensor has the highest mean temperature across its valid readings?
- What is the 90th percentile AQI across all valid readings? Use `np.nanpercentile`.
- Filter to `'normal'` status readings where both `temp_c` and `aqi` are valid. Is there a correlation between temperature and AQI? Use `np.corrcoef`.

In [48]:
sensors = pd.DataFrame({
    'sensor':      ['S1','S2','S3','S4'] * 3,
    'month':       ['Jan']*4 + ['Feb']*4 + ['Mar']*4,
    'temp_c':      [21.3, -999, 19.8, 22.5,  23.1, 20.6, -999, 23.4,  22.0, 19.9, 18.7, -999],
    'aqi':         [48, 65, 'ERR', 52,  55, 58, 71, 'ERR',  43, 62, 'ERR', 49],
    'status_code': [1, 1, 2, 3,  1, 2, 1, 3,  1, 1, 2, 3],
})

# Your code here

sensors['status_code'] = sensors['status_code'].replace({1:'normal',2:'maintenance',3:'offline'})
sensors['temp_c'] = sensors['temp_c'].replace({-999:np.nan})
sensors['aqi'] = sensors['aqi'].replace({'ERR':np.nan})

g = sensors.groupby('sensor')['temp_c'].apply(lambda x: np.nanmean(x))
print(g.idxmax(),'has the highest mean readings')
print(np.nanpercentile(sensors['aqi'],90), 'is the 90th percentile')
f = sensors.dropna(subset=['temp_c','aqi'], how='any')
f_norm = f.loc[f['status_code'] == 'normal']

cors = np.corrcoef(f['temp_c'], f['aqi'])[0,1]
print('correlation is: ', cors)
print('there is a negative correlation')

S4 has the highest mean readings
66.2 is the 90th percentile
correlation is:  -0.5023995609054691
there is a negative correlation


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_90639/2569649136.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sensors['aqi'] = sensors['aqi'].replace({'ERR':np.nan})


---

## Level 3 — Freelancer payment records

A billing export from a freelance platform. The data has these problems — **no cleaning steps are provided; design the pipeline yourself**:

- `payment_id`: one record appears twice (exact duplicate)
- `client`: inconsistent casing (`"Acme Corp"`, `"acme corp"`, `"ACME CORP"`)
- `amount`: stored as strings like `"$1,200"` — strip symbols and commas, convert to float
- `status`: inconsistent casing
- `pay_date`: some entries are `"TBD"` — use `pd.to_datetime(..., errors='coerce')` so they become `NaT`

Build a `.pipe()` chain with at least two functions. Write the functions yourself.

After cleaning:

1. Which client has the highest total amount where `status == 'paid'`?
2. Log-transform all amounts with `np.log`. What is the mean and standard deviation of log-amounts across all records? (`np.log` compresses a wide dollar range into a more comparable scale.)
3. Group by client and sum total amount (all statuses). Use `np.argsort` to rank clients from highest to lowest total.
4. What fraction of payments have no confirmed date (`NaT`)?

In [60]:
payments = pd.DataFrame({
    'payment_id': ['P001','P002','P003','P004','P005','P006','P007','P008',
                   'P009','P010','P011','P012','P013','P014','P015','P003'],
    'client':     ['Acme Corp','Globex','acme corp','GLOBEX','Initech','ACME CORP',
                   'Globex','initech','Acme Corp','Initech','GLOBEX','acme corp',
                   'Initech','Globex','ACME CORP','acme corp'],
    'amount':     ['$1,200','$850','$2,400','$1,100','$750','$3,200','$920','$1,650',
                   '$480','$2,100','$780','$1,350','$640','$1,080','$2,900','$2,400'],
    'status':     ['paid','PAID','pending','Paid','PENDING','paid','failed','Paid',
                   'paid','pending','PAID','failed','pending','Paid','paid','pending'],
    'pay_date':   ['2024-01-15','2024-01-22','2024-02-05','2024-02-18','TBD',
                   '2024-03-01','2024-03-14','2024-03-28','2024-04-09','TBD',
                   '2024-04-20','2024-05-03','TBD','2024-05-18','2024-05-29','2024-02-05'],
})

# Your code here

def clean(df):
    df = df.copy()
    df = df.drop_duplicates()
    return df
def convert(df):
    df['client'] = df['client'].str.lower()
    df['amount'] = df['amount'].str.replace('$','',regex = False)
    df['amount'] = df['amount'].str.replace(',','',regex = False)
    df['amount'] = pd.to_numeric(df['amount'])
    df['status'] = df['status'].str.lower()
    df['pay_date'] = pd.to_datetime(df['pay_date'], errors= 'coerce')
    return df

cp = payments.copy().pipe(clean).pipe(convert)
cpf = cp.loc[cp['status'] == 'paid']
g = cpf.groupby('client')['amount'].sum()
print(g.idxmax(), 'has the hightest total amount')
cp['log_amount'] = np.log(cp['amount'])
print('mean is', cp['log_amount'].mean(), 'std is',cp['log_amount'].std())
ga = cp.groupby('client')['amount'].sum()
print(ga.index[np.argsort(-ga)])

print(np.mean(np.isnat(cp['pay_date'])),'have no confirmed date')


acme corp has the hightest total amount
mean is 7.11031794825433 std is 0.5664929359872513
Index(['acme corp', 'initech', 'globex'], dtype='object', name='client')
0.2 have no confirmed date



1. Which client has the highest total amount where `status == 'paid'`?
2. Log-transform all amounts with `np.log`. What is the mean and standard deviation of log-amounts across all records? (`np.log` compresses a wide dollar range into a more comparable scale.)
3. Group by client and sum total amount (all statuses). Use `np.argsort` to rank clients from highest to lowest total.
4. What fraction of payments have no confirmed date (`NaT`)?